[Reference](https://ai.plainenglish.io/from-messy-text-to-interactive-visualizations-with-google-langextract-060e53ffa9da)

# Step 1: Define the Prompt and Extraction Rules

In [1]:
# Step 1: Define the prompt and extraction rules
prompt = textwrap.dedent(f"""
        You are an expert information extractor. Your task is to pull out
        relevant entities and relationships from the given text.

        Focus on: {query}

        Extract entities with their types and meaningful attributes.
        Identify relationships between entities with clear connection types.
        Use exact text for extractions. Do not paraphrase or overlap entities.
    """)

# Step 2: Few-Shot Examples

In [2]:
# templates/few_shot_examples.py

import langextract as lx

def get_dynamic_examples(query: str) -> list:
    """Return appropriate few-shot examples based on query keywords"""

    if any(keyword in query.lower() for keyword in ['financial', 'revenue', 'company', 'business', 'founder', 'ceo']):
        # Business-focused examples with relationships
        return [
            lx.data.ExampleData(
                text="Apple Inc. reported $394.3 billion in revenue for fiscal 2022. The company is headquartered in Cupertino, California. Steve Jobs founded Apple Inc.",
                extractions=[
                    lx.data.Extraction(
                        extraction_class="company",
                        extraction_text="Apple Inc.",
                        attributes={"type": "technology_company"}
                    ),
                    lx.data.Extraction(
                        extraction_class="financial_metric",
                        extraction_text="$394.3 billion in revenue",
                        attributes={"period": "fiscal 2022", "metric_type": "revenue"}
                    ),
                    lx.data.Extraction(
                        extraction_class="location",
                        extraction_text="Cupertino, California",
                        attributes={"type": "headquarters"}
                    ),
                    lx.data.Extraction(
                        extraction_class="person",
                        extraction_text="Steve Jobs",
                        attributes={"role": "founder"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="Apple Inc. is headquartered in Cupertino, California",
                        attributes={"type": "located_in", "subject": "Apple Inc.", "object": "Cupertino, California"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="Steve Jobs founded Apple Inc.",
                        attributes={"type": "founded_by", "subject": "Steve Jobs", "object": "Apple Inc."}
                    ),
                ]
            )
        ]
    elif any(keyword in query.lower() for keyword in ['legal', 'contract']):
        # Legal-focused examples
        return [
            lx.data.ExampleData(
                text="The agreement between XYZ Corp and ABC Ltd was signed on January 15, 2024, with a term of 5 years.",
                extractions=[
                    lx.data.Extraction(
                        extraction_class="party",
                        extraction_text="XYZ Corp",
                        attributes={"role": "contractor"}
                    ),
                    lx.data.Extraction(
                        extraction_class="party",
                        extraction_text="ABC Ltd",
                        attributes={"role": "client"}
                    ),
                    lx.data.Extraction(
                        extraction_class="date",
                        extraction_text="January 15, 2024",
                        attributes={"event": "signing_date"}
                    ),
                    lx.data.Extraction(
                        extraction_class="term",
                        extraction_text="5 years",
                        attributes={"duration": "5 years"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="XYZ Corp signed agreement with ABC Ltd",
                        attributes={"type": "contractual_agreement", "subject": "XYZ Corp", "object": "ABC Ltd"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="Agreement was signed on January 15, 2024",
                        attributes={"type": "signed_on", "subject": "agreement", "object": "January 15, 2024"}
                    ),
                ]
            )
        ]
    else:
        # Generic example with relationships - diverse domain
        return [
            lx.data.ExampleData(
                text="Romeo loved Juliet deeply, despite their families' feud in Verona. They met at the Capulet party.",
                extractions=[
                    lx.data.Extraction(
                        extraction_class="character",
                        extraction_text="Romeo",
                        attributes={"role": "protagonist", "family": "Montague"}
                    ),
                    lx.data.Extraction(
                        extraction_class="character",
                        extraction_text="Juliet",
                        attributes={"role": "protagonist", "family": "Capulet"}
                    ),
                    lx.data.Extraction(
                        extraction_class="location",
                        extraction_text="Verona",
                        attributes={"type": "city"}
                    ),
                    lx.data.Extraction(
                        extraction_class="event",
                        extraction_text="Capulet party",
                        attributes={"type": "social_gathering"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="Romeo loved Juliet",
                        attributes={"type": "romantic_love", "subject": "Romeo", "object": "Juliet"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="Romeo and Juliet met at the Capulet party",
                        attributes={"type": "met_at", "subject": "Romeo and Juliet", "object": "Capulet party"}
                    ),
                    lx.data.Extraction(
                        extraction_class="relationship",
                        extraction_text="families' feud in Verona",
                        attributes={"type": "conflict", "subject": "Montague and Capulet families", "object": "Verona"}
                    ),
                ]
            )
        ]

# Step 3: Run the Extraction

In [3]:
result = lx.extract(
            text_or_documents=text,
            prompt_description=prompt,
            examples=examples,
            model_id="gemini-2.5-flash",
            api_key=os.getenv("GOOGLE_API_KEY"),
            # max_workers=5,  # Reasonable number of workers
            # extraction_passes=2  # Two passes for better recall
            )

# Step 4: LangExtract’s native HTML visualization


In [4]:
# src/display_manager.py
import streamlit as st
import streamlit.components.v1 as components
import langextract as lx
import os
from typing import List, Dict, Any

def create_highlighted_html(extraction_results: List, output_dir: str = "./data/outputs"):
    """Generate LangExtract's native HTML visualization"""

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    try:
        # Save extraction results to JSONL
        output_file = os.path.join(output_dir, "extraction_results.jsonl")
        lx.io.save_annotated_documents(extraction_results,
                                       output_name="extraction_results.jsonl",
                                       output_dir=output_dir)

        # Generate HTML visualization
        html_content = lx.visualize(output_file)

        # Save HTML file
        html_file = os.path.join(output_dir, "visualization.html")
        with open(html_file, "w", encoding='utf-8') as f:
            if hasattr(html_content, 'data'):
                f.write(html_content.data)  # For Jupyter/Colab environments
            else:
                f.write(str(html_content))

        return html_file, html_content

    except Exception as e:
        st.error(f"Error generating visualization: {str(e)}")
        return None, None

def render_entity_highlights(html_content: str):
    """Display the LangExtract visualization in Streamlit"""

    if html_content is None:
        st.error("No visualization content available")
        return

    try:
        # Extract just the content if it's wrapped
        if hasattr(html_content, 'data'):
            html_to_display = html_content.data
        else:
            html_to_display = str(html_content)

        # Display in Streamlit using components
        components.html(html_to_display, height=800, scrolling=True)

    except Exception as e:
        st.error(f"Error displaying visualization: {str(e)}")

# Step 5: Making It Interactive with Streamlit

In [5]:
import streamlit as st
import os
from typing import Dict, Any
from src.utils import load_gemini_key
from src.entity_extractor import extract_entities_from_documents
from src.display_manager import create_highlighted_html, render_entity_highlights, show_extraction_summary
from data.sample_documents import SAMPLE_DOCUMENTS

def main():
    """Main Streamlit application"""

    # Page configuration
    st.set_page_config(
        page_title="LangExtract Knowledge Extraction",
        page_icon="🔍",
        layout="wide"
    )

    st.title("🔍 LangExtract Knowledge Extraction")
    st.markdown("Transform unstructured text into highlighted, structured information using Google's LangExtract")

    # Load API key
    api_key, key_provided = load_gemini_key()

    if not key_provided:
        st.warning("⚠️ Please provide a Gemini API key to continue")
        st.stop()

    # Set environment variable for LangExtract
    os.environ["GOOGLE_API_KEY"] = api_key

    # Display sample documents info
    st.success(f"📚 Ready to process {len(SAMPLE_DOCUMENTS)} documents")

    # User query input
    user_query = st.text_input(
        "🔍 Enter your query (optional):",
        placeholder="e.g., 'company founders and locations' or 'financial information'"
    )

    # Process documents button
    if st.button("🚀 Process Documents & Extract Information", type="primary"):
        with st.spinner("Processing documents..."):
            try:
                results = extract_entities_from_documents(SAMPLE_DOCUMENTS, user_query)

                # Display results in simplified tabs
                tab1, tab2, tab3 = st.tabs(["📋 Highlighted Text", "📊 Entity Summary", "🔍 Search Results"])

                with tab1:
                    st.subheader("📋 Text with Highlighted Entities")

                    # Generate and display LangExtract's native visualization
                    try:
                        html_file, html_content = create_highlighted_html(
                            results["langextract_results"]
                        )

                        st.info("💡 Entities are highlighted directly in the source text below:")
                        render_entity_highlights(html_content)

                        # Provide download link for the HTML file
                        with open(html_file, 'r', encoding='utf-8') as f:
                            st.download_button(
                                "📥 Download Full Visualization",
                                f.read(),
                                "langextract_visualization.html",
                                "text/html"
                            )

                    except Exception as e:
                        st.error(f"Visualization error: {str(e)}")
                        st.info("Showing entity summary instead:")
                        show_extraction_summary(results["entities"])

                with tab2:
                    st.subheader("📊 Extraction Summary")
                    show_extraction_summary(results["entities"])

                    # Optional: Show raw extraction data
                    with st.expander("🔧 Raw Extraction Data"):
                        st.json(results["entities"])

                with tab3:
                    st.subheader("🔍 Query-Specific Results")
                    if user_query:
                        # Simple query matching
                        query_words = user_query.lower().split()
                        matching_entities = []

                        for entity in results["entities"]:
                            entity_text = entity["text"].lower()
                            entity_attrs = str(entity.get("attributes", {})).lower()

                            if any(word in entity_text or word in entity_attrs for word in query_words):
                                matching_entities.append(entity)

                        st.metric("Matching Entities", len(matching_entities))

                        if matching_entities:
                            for entity in matching_entities:
                                with st.expander(f"{entity['class'].title()}: {entity['text']}"):
                                    st.write(f"**Type:** {entity['class']}")
                                    if entity.get('attributes'):
                                        st.write("**Attributes:**")
                                        for key, value in entity['attributes'].items():
                                            st.write(f"- {key}: {value}")
                        else:
                            st.info("No entities match your query. Try different keywords.")
                    else:
                        st.info("Enter a query above and reprocess to see filtered results.")

            except Exception as e:
                st.error(f"Processing failed: {str(e)}")
                st.error("Please check your API key and internet connection.")

if __name__ == "__main__":
    main()